## Extract channels and merge the data

In [7]:
import os
import glob
import pandas as pd


# =========================
# 1. Path configuration
# =========================
input_dir = r"D:/MyProjects/EEGDatasets/EPOCX/data"
extracted_dir = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted"
merged_output = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_merged/all_merged.csv"

os.makedirs(extracted_dir, exist_ok=True)
os.makedirs(os.path.dirname(merged_output), exist_ok=True)


# =========================
# 2. Label mapping
#    key: file name without extension, or file name prefix
#    value: class label
# =========================
label_map = {
    "gaming3": "concentrate",
    "resting_ec": "relax",
}


# =========================
# 3. Frontal channels and bands
# =========================
frontal_channels = ["AF3", "F7", "F3", "FC5", "FC6", "F4", "F8", "AF4"]
bands = ["Theta", "Alpha", "BetaL", "BetaH", "Gamma"]

selected_columns = [
    f"POW.{ch}.{band}"
    for ch in frontal_channels
    for band in bands
]


# =========================
# 4. Get label from file name
#    Return None if no label is found
# =========================
def get_label_from_filename(file_path, label_map):
    base_name = os.path.splitext(os.path.basename(file_path))[0]

    if base_name in label_map:
        return label_map[base_name]

    for prefix, label in label_map.items():
        if base_name.startswith(prefix):
            return label

    return None


# =========================
# 5. Extract frontal band power from one file
# =========================
def extract_frontal_pow(csv_path, output_dir, label_map):
    df = pd.read_csv(csv_path, skiprows=1)

    label_value = get_label_from_filename(csv_path, label_map)
    if label_value is None:
        print(f"Skipped: {os.path.basename(csv_path)} (no label found)")
        return None

    missing_cols = [col for col in selected_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing columns in {os.path.basename(csv_path)}:\n" + "\n".join(missing_cols)
        )

    df_out = df[selected_columns].copy()
    df_out["label"] = label_value

    base_name = os.path.splitext(os.path.basename(csv_path))[0]
    output_path = os.path.join(output_dir, f"{base_name}_frontal.csv")
    df_out.to_csv(output_path, index=False, encoding="utf-8-sig")

    return output_path


# =========================
# 6. Batch extraction
# =========================
all_csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
extracted_files = []

for csv_file in all_csv_files:
    try:
        out_file = extract_frontal_pow(csv_file, extracted_dir, label_map)
        if out_file is not None:
            extracted_files.append(out_file)
            print(f"Processed: {os.path.basename(csv_file)} -> {os.path.basename(out_file)}")
    except Exception as e:
        print(f"Failed: {os.path.basename(csv_file)}")
        print(e)


# =========================
# 7. Merge all extracted files
# =========================
if extracted_files:
    merged_df = pd.concat([pd.read_csv(f) for f in extracted_files], ignore_index=True)
    merged_df.to_csv(merged_output, index=False, encoding="utf-8-sig")
    print(f"\nMerged file saved to: {merged_output}")
    print(f"Merged shape: {merged_df.shape}")
else:
    print("No extracted files to merge.")

Skipped: gaming1.csv (no label found)
Skipped: gaming2.csv (no label found)
Processed: gaming3.csv -> gaming3_frontal.csv
Processed: resting_ec.csv -> resting_ec_frontal.csv
Skipped: resting_eo.csv (no label found)
Skipped: rest_eo_postgame.csv (no label found)


In [2]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================
# 1. Path configuration
# =========================

output_dir = os.path.dirname(merged_output)
os.makedirs(output_dir, exist_ok=True)

results_path = os.path.join(output_dir, "results_logreg.txt")
save_model = os.path.join(output_dir, "epocx_logreg_model.joblib")
save_scaler = os.path.join(output_dir, "epocx_scaler.joblib")
save_meta = os.path.join(output_dir, "epocx_feature_names.joblib")


# =========================
# 2. Read merged data
# =========================
df = pd.read_csv(merged_output)

print("Data shape:", df.shape)
print(df.head())


# =========================
# 3. Separate features and labels
# =========================
X = df.drop(columns=["label"])
y = df["label"]

feature_names = X.columns.tolist()

print("\nLabel distribution:")
print(y.value_counts())


# =========================
# 4. Split train / test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# 5. Standardization
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================
# 6. Logistic Regression
# =========================
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)


# =========================
# 7. Prediction
# =========================
y_pred = model.predict(X_test)


# =========================
# 8. Evaluation
# =========================
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\nAccuracy:", acc)
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(cm)


# =========================
# 9. Save results
# =========================
with open(results_path, "w", encoding="utf-8") as f:
    f.write(f"Merged data path: {merged_output}\n")
    f.write(f"Data shape: {df.shape}\n\n")

    f.write("Label distribution:\n")
    f.write(y.value_counts().to_string())
    f.write("\n\n")

    f.write(f"Accuracy: {acc}\n\n")

    f.write("Classification Report:\n")
    f.write(report)
    f.write("\n")

    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n")

print(f"\nSaved results to {results_path}")


# =========================
# 10. Save model and preprocessing files
# =========================
joblib.dump(model, save_model)
joblib.dump(scaler, save_scaler)
joblib.dump(feature_names, save_meta)

print(f"Saved model to {save_model}")
print(f"Saved scaler to {save_scaler}")
print(f"Saved feature names to {save_meta}")

Data shape: (3364, 41)
   POW.AF3.Theta  POW.AF3.Alpha  POW.AF3.BetaL  POW.AF3.BetaH  POW.AF3.Gamma  \
0       4.456561       3.495647       0.839779       0.325413       0.365118   
1       4.559149       2.995620       0.647689       0.346526       0.353810   
2       4.543970       2.642398       0.543687       0.388384       0.341476   
3       4.526125       2.499658       0.570819       0.450692       0.334778   
4       4.524489       2.525326       0.731554       0.528926       0.334570   

   POW.F7.Theta  POW.F7.Alpha  POW.F7.BetaL  POW.F7.BetaH  POW.F7.Gamma  ...  \
0      3.269726      1.190943      1.039884      0.253717      0.246723  ...   
1      3.316445      1.317984      0.916580      0.253443      0.233739  ...   
2      3.418697      1.674606      0.825198      0.278880      0.236419  ...   
3      3.412848      2.231747      0.802202      0.325729      0.259437  ...   
4      3.224709      2.861127      0.851462      0.385452      0.300356  ...   

   POW.F8.Alpha

In [11]:
import os
import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =========================
# 1. Path configuration
# =========================
input_dir = r"D:/MyProjects/EEGDatasets/EPOCX/data"
extracted_dir = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted"
merged_output = r"D:/MyProjects/EEGDatasets/EPOCX/EPOCX_merged/all_merged.csv"

model_dir = os.path.dirname(merged_output)

model_path = os.path.join(model_dir, "epocx_logreg_model.joblib")
scaler_path = os.path.join(model_dir, "epocx_scaler.joblib")
feature_path = os.path.join(model_dir, "epocx_feature_names.joblib")

# file to evaluate
test_file = os.path.join(extracted_dir, "resting_ec_frontal.csv")


# =========================
# 2. Load model artifacts
# =========================
model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
feature_names = joblib.load(feature_path)


# =========================
# 3. Read new data
# =========================
df_new = pd.read_csv(test_file)

print("Test file:", test_file)
print("Data shape:", df_new.shape)
print(df_new.head())


# =========================
# 4. Prepare features and labels
# =========================
X_new = df_new[feature_names]
y_true = df_new["label"]


# =========================
# 5. Standardize and predict
# =========================
X_new_scaled = scaler.transform(X_new)
y_pred = model.predict(X_new_scaled)

print("\nPredictions:")
print(y_pred[:10])


# =========================
# 6. Evaluate
# =========================
acc = accuracy_score(y_true, y_pred)
all_labels = model.classes_

cm = confusion_matrix(y_true, y_pred, labels=all_labels)
report = classification_report(
    y_true,
    y_pred,
    labels=all_labels,
    target_names=all_labels,
    zero_division=0
)

print("\nAccuracy:", acc)

print("\nClassification Report:")
print(report)

print("\nConfusion Matrix:")
print(cm)

Test file: D:/MyProjects/EEGDatasets/EPOCX/EPOCX_extracted\resting_ec_frontal.csv
Data shape: (962, 41)
   POW.AF3.Theta  POW.AF3.Alpha  POW.AF3.BetaL  POW.AF3.BetaH  POW.AF3.Gamma  \
0       6.635341      32.436275       0.783854       1.419582       0.495725   
1       7.645335      39.225010       0.896968       1.745917       0.536353   
2       7.899293      47.229721       1.029868       2.055595       0.569899   
3       7.250029      55.532196       1.205022       2.260139       0.588381   
4       5.992597      62.627327       1.386965       2.302165       0.585091   

   POW.F7.Theta  POW.F7.Alpha  POW.F7.BetaL  POW.F7.BetaH  POW.F7.Gamma  ...  \
0      2.954771     15.932360      0.707922      1.292939      0.396605  ...   
1      3.008000     21.093550      0.894737      1.550869      0.373699  ...   
2      2.942324     28.201450      1.136394      1.778938      0.340112  ...   
3      2.714083     36.044971      1.399838      1.901475      0.301316  ...   
4      2.349721